# Proyecto 4 — Clasificacion BPC Estacion de Bombeo

**Notebook final auditable** — documenta resultados ya generados por el pipeline y el dashboard.  
No reentrena modelos, no recalcula SHAP, PERMANOVA/PERMDISP ni assessment; solo **lee** tablas y figuras exportadas.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

# Raiz del subproyecto (detecta run_pipeline.py subiendo desde cwd)
_p = Path.cwd().resolve()
PROJECT_ROOT = _p
for _ in range(6):
    if (PROJECT_ROOT / "run_pipeline.py").is_file():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    PROJECT_ROOT = _p

OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figures"
OUTPUTS_MODELS = PROJECT_ROOT / "outputs" / "models"
DASHBOARD_DATA = PROJECT_ROOT / "data" / "dashboard"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"


def load_csv(rel: Path, *, required: bool = True) -> pd.DataFrame:
    if not rel.is_file():
        if required:
            display(Markdown(f"**Falta archivo requerido:** `{rel.relative_to(PROJECT_ROOT)}`"))
        return pd.DataFrame()
    return pd.read_csv(rel, encoding="utf-8")


def display_table(df: pd.DataFrame, max_rows: int = 20) -> None:
    if df.empty:
        display(Markdown("_Sin datos._"))
        return
    display(df.head(max_rows))


def display_figure(rel: Path, width=None):
    if rel.is_file():
        w = 900 if width is None else width
        display(Image(filename=str(rel), width=w))
    else:
        display(Markdown(f"_Figura no encontrada: `{rel.relative_to(PROJECT_ROOT)}`_"))


def get_kpi(kpis_df: pd.DataFrame, metric: str):
    if kpis_df.empty or "metric" not in kpis_df.columns:
        return None
    m = kpis_df.loc[kpis_df["metric"].astype(str) == metric, "value"]
    return m.iloc[0] if len(m) else None



## 1. Resumen ejecutivo

- **Problema:** clasificar el tipo de operacion/crudo trasegado en una BPC (bomba centrifuga) a partir de **firma vibratoria** (24 sensores), usando la etiqueta `Batch`.
- **Objetivo:** modelo supervisado multicase **CASTILLA**, **MEZCLA**, **RUBIALES** con validacion temporal.
- **Dataset:** serie temporal de alta frecuencia agregada a **ventanas de 60 s**; **MEZCLA** es **clase operacional independiente** (no composicion quimica explicita).
- **Mejor modelo (CV):** ver `best_model_name` en KPIs (tipicamente XGBoost).
- **Metricas de generalizacion:** **validacion cruzada temporal** (F1 macro, balanced accuracy, MCC, Kappa); **no** usar `training_f1_macro` del entrenamiento final como generalizacion.
- **Hallazgos estadisticos:** separacion de firmas entre batches (PERMANOVA) y diferencias de dispersion (PERMDISP); PCA como visualizacion.
- **Interpretabilidad:** permutation importance y SHAP global — **no implican causalidad**.
- **Assessment:** pesos ponderados del activo — **no son importancia ML**; metodo dominante **robust_percentile_fallback** (umbrales H, HH, V0 incompletos en Excel).
- **Uso operacional:** no implementar en **control real** sin revision tecnica.
- **Dashboard:** aplicacion Dash en `dashboard/app.py`, solo CSV en `data/dashboard/`.

In [ ]:
kpis = load_csv(DASHBOARD_DATA / "dashboard_kpis.csv", required=True)
wanted = [
    "n_rows_raw", "n_windows_modeling", "n_model_features", "permanova_r2", "permanova_p_value",
    "permdisp_p_value", "best_model_name", "best_model_f1_macro_cv", "best_model_balanced_accuracy_cv",
    "top_component", "top_family", "condition_index_mean", "assessment_method",
]
rows = []
for m in wanted:
    rows.append({"metric": m, "value": get_kpi(kpis, m)})
kpi_tbl = pd.DataFrame(rows)
display(Markdown("### KPIs clave (desde `data/dashboard/dashboard_kpis.csv`)"))
display(kpi_tbl)


## 2. Contexto del activo y del dataset

La estacion incluye una **BPC (bomba centrifuga)** con **motor** y **variador**. Se registran **24 variables vibracionales** (desplazamiento, aceleracion RMS, velocidad RMS en distintos ejes y ubicaciones).

Cada fila del dataset crudo tiene **Timestamp** y **Batch** (clase operacional). Clases: **CASTILLA**, **MEZCLA**, **RUBIALES**. **MEZCLA** se modela como **clase independiente**; no se afirma que sea una mezcla fisica de Castilla y Rubiales.

**Limitacion:** no hay en este proyecto variables de proceso (presion, caudal, densidad, temperatura de fluido) — solo vibracion.

In [ ]:
display(Markdown("### Transiciones de batch (dashboard)"))
display_table(load_csv(DASHBOARD_DATA / "dashboard_batch_transitions.csv", required=False), 20)
display(Markdown("### Distribucion de clases (serie cruda agregada / quality)"))
display_table(load_csv(OUTPUTS_TABLES / "class_distribution.csv"), 10)


## 3. Calidad del dato

Resumen de calidad y ausencia/presencia de missing, gaps temporales y transiciones entre batches.

In [ ]:
dq = load_csv(OUTPUTS_TABLES / "data_quality_summary.csv")
display_table(dq, 30)
miss = load_csv(OUTPUTS_TABLES / "missing_values.csv", required=False)
if not miss.empty and miss["missing_count"].sum() > 0:
    display(Markdown("### Missing values (solo si hay faltantes)"))
    display_table(miss[miss["missing_count"] > 0], 30)
else:
    display(Markdown("### Missing values: **0** faltantes totales segun `missing_values.csv`."))
gaps = load_csv(OUTPUTS_TABLES / "timestamp_gaps.csv", required=False)
if not gaps.empty:
    display(Markdown("### Gaps temporales"))
    display_table(gaps, 20)
else:
    display(Markdown("### Gaps temporales: **0** filas en `timestamp_gaps.csv`."))
display(Markdown("### Transiciones (outputs)"))
bt = load_csv(OUTPUTS_TABLES / "batch_transitions.csv", required=False)
display_table(bt, 20)
if not bt.empty:
    display(Markdown(f"**Transiciones detectadas:** {len(bt)} (según `outputs/tables/batch_transitions.csv`)."))
display(Markdown("**Nota:** las ventanas cercanas a transiciones se excluyen del modelado; conviene interpretar con cautela los cruces de clase."))


## 4. Ventaneo temporal y feature engineering

- No se modela **fila a fila** por no i.i.d. y por coste; se usa **ventana de 60 s**.
- **24 variables** x **5 estadisticos** (median, mean, iqr, p95, p05) = **120 features** por ventana.
- Se descartan ventanas **ambigues** y, para modelado, ventanas **cerca de transicion** (vecindario de buffer).

**Advertencia metodologica:** las exclusiones afectan el conteo de ventanas usadas vs totales.

In [ ]:
display_table(load_csv(OUTPUTS_TABLES / "windowed_dataset_summary.csv"), 10)
wd = load_csv(OUTPUTS_TABLES / "windowed_class_distribution.csv", required=False)
if not wd.empty:
    display(Markdown("### Distribucion de clases en ventanas"))
    display_table(wd, 20)
else:
    display(Markdown("_No se encontro `windowed_class_distribution.csv`._"))


## 5. Validacion estadistica de separabilidad

- **PERMANOVA:** contraste de **localizacion / separacion de centroides** (diferencias de posicion central de la firma multivariante entre batches).
- **PERMDISP:** contraste de **dispersion interna** (heterogeneidad dentro del batch; complementa PERMANOVA pero no mide lo mismo que el centroide).
- **PCA:** visualizacion; no sustituye inferencia formal.

**Interpretacion:** PERMANOVA significativo sugiere firmas **diferenciables** entre batches. PERMDISP significativo sugiere **dispersion interna distinta**. **No** inferir composicion quimica del crudo desde vibracion.

In [ ]:
display_table(load_csv(OUTPUTS_TABLES / "statistical_validation_summary.csv"), 5)
display(Markdown("### Pairwise PERMANOVA (FDR)"))
display_table(load_csv(OUTPUTS_TABLES / "pairwise_permanova_fdr_results.csv"), 10)
display(Markdown("### Pairwise PERMDISP (FDR)"))
display_table(load_csv(OUTPUTS_TABLES / "pairwise_permdisp_fdr_results.csv"), 10)
display(Markdown("### Centroides PCA"))
display_table(load_csv(OUTPUTS_TABLES / "pca_centroids.csv"), 10)
display(Markdown("### Figuras"))
display_figure(OUTPUTS_FIGURES / "pca_batches.png")
display_figure(OUTPUTS_FIGURES / "permdisp_boxplot.png")
display_figure(OUTPUTS_FIGURES / "top_kruskal_features.png")


## 6. Modelado comparativo

Modelos comparados: **logistic_regression**, **svm_rbf**, **random_forest**, **xgboost**, **soft_voting_ensemble**.  
Validacion **temporal** con **StratifiedGroupKFold** por grupos (ver `cv_split_summary.csv`).

**Metricas:** F1 macro, balanced accuracy, MCC, Cohen Kappa (promedios CV en `dashboard_model_metrics.csv`).

**Hallazgos:** el modelo con mayor F1 macro en CV suele ser **XGBoost**; la ventaja frente a **soft_voting_ensemble** suele ser **marginal**. **No** usar `training_f1_macro` ni otras metricas del entrenamiento final sobre todas las ventanas como generalizacion.

In [ ]:
display_table(load_csv(DASHBOARD_DATA / "dashboard_model_metrics.csv"), 10)
display(Markdown("### Matriz de confusion (mejor modelo, OOF agregado)"))
display_table(load_csv(DASHBOARD_DATA / "dashboard_confusion_best_model.csv"), 15)
display_table(load_csv(OUTPUTS_TABLES / "cv_split_summary.csv", required=False), 20)
display_figure(OUTPUTS_FIGURES / "model_comparison_f1_macro.png")
display_figure(OUTPUTS_FIGURES / "model_comparison_balanced_accuracy.png")
display_figure(OUTPUTS_FIGURES / "confusion_matrix_best_model.png")


## 7. Modelo final

El modelo final (**XGBoost**) se entrena sobre las **1451** ventanas validas (sin transicion), con **120** features. Artefactos en `outputs/models/` (por ejemplo `model_final_xgboost.pkl`).

**Diferencia clave:** filas de `final_model_training_summary.csv` reflejan **ajuste en todo el conjunto de entrenamiento**; las metricas de **generalizacion** del notebook ejecutivo deben tomarse de **CV** (`dashboard_model_metrics.csv`).

In [ ]:
display_table(load_csv(OUTPUTS_TABLES / "final_model_training_summary.csv"), 5)
display(Markdown("### Primeras filas de predicciones finales"))
display_table(load_csv(OUTPUTS_TABLES / "final_model_predictions.csv"), 12)
display(Markdown("**Advertencia:** `training_f1_macro = 1.0` (si aparece) es sobre entrenamiento completo, **no** es metrica de generalizacion."))


## 8. Interpretabilidad del modelo

- **Permutation importance** y **SHAP** global ayudan a explicar **contribuciones al modelo**; **no** demuestran causalidad ni mecanismo fisico.
- Rankings agregados por **componente**, **familia**, **posicion** y **estadistico**.

**Lectura tipica:** componente **bomba** suele liderar importancia agregada, seguido de cerca por **motor**; familia **acceleration_rms** suele ser la mas relevante. La mejor feature consolidada aparece en `interpretability_summary.csv` (`best_feature_consolidated`).

In [ ]:
display_table(load_csv(OUTPUTS_TABLES / "interpretability_summary.csv"), 5)
fi = load_csv(DASHBOARD_DATA / "dashboard_feature_importance.csv")
if not fi.empty and "consolidated_score" in fi.columns:
    display(Markdown("### Top 20 features (consolidated_score)"))
    display_table(fi.sort_values("consolidated_score", ascending=False).head(20), 20)
for name, title in [
    ("dashboard_rank_component.csv", "Rank componente"),
    ("dashboard_rank_family.csv", "Rank familia"),
    ("dashboard_rank_position.csv", "Rank posicion"),
    ("dashboard_rank_statistic.csv", "Rank estadistico"),
]:
    display(Markdown(f"### {title}"))
    display_table(load_csv(DASHBOARD_DATA / name), 15)
display_figure(OUTPUTS_FIGURES / "top_20_permutation_importance.png")
display_figure(OUTPUTS_FIGURES / "top_20_shap_importance.png")
display_figure(OUTPUTS_FIGURES / "importance_by_component.png")
display_figure(OUTPUTS_FIGURES / "importance_by_family.png")
display_figure(OUTPUTS_FIGURES / "importance_by_position.png")
display_figure(OUTPUTS_FIGURES / "importance_by_statistic.png")


## 9. Pesos ponderados y assessment de condicion

Los **pesos ponderados** definen un **indice de condicion del activo** (umbrales / percentiles), **no** la importancia ML de las variables en el clasificador.

Se esperan **24/24** variables mapeadas (ver `assessment_summary.csv`). Con **H, HH y V0** vacios en el Excel, el metodo dominante es **robust_percentile_fallback**.

El indice es **exploratorio y comparativo** entre ventanas/batches; **no** es diagnostico normativo de falla.

In [ ]:
display_table(load_csv(OUTPUTS_TABLES / "assessment_summary.csv"), 5)
display_table(load_csv(DASHBOARD_DATA / "dashboard_condition_index_by_batch.csv"), 10)
display_table(load_csv(DASHBOARD_DATA / "dashboard_top_weighted_variables.csv"), 25)
display_figure(OUTPUTS_FIGURES / "condition_index_by_batch.png")
display_figure(OUTPUTS_FIGURES / "condition_index_time_series.png")
display_figure(OUTPUTS_FIGURES / "top_weighted_variables.png")


## 10. Dashboard

Aplicacion **Dash** en `dashboard/app.py`: lee **solo** CSV de `data/dashboard/`, **no** reentrena modelos ni recalcula pipeline. **Siete pestanas:** resumen ejecutivo; desempeno del modelo; predicciones por ventana; validacion estadistica; interpretabilidad; assessment ponderado; datos y advertencias.

**Comandos:**
```text
python run_pipeline.py --stage dashboard_exports
python dashboard/app.py
```
Luego abrir `http://127.0.0.1:8050`.

In [ ]:
readme = DASHBOARD_DATA / "README_dashboard_data.md"
if readme.is_file():
    txt = readme.read_text(encoding="utf-8")
    display(Markdown("### Extracto `README_dashboard_data.md`"))
    display(Markdown(txt[:2500] + ("\n\n..." if len(txt) > 2500 else "")))
else:
    display(Markdown("_README no encontrado._"))


## 11. Limitaciones

- Campana de **~24 h** de datos; resultados dependen de esa ventana operativa.
- **Sin variables de proceso** (presion, caudal, densidad, temperatura).
- **No** implementar en **control real** sin revision tecnica y pruebas en planta.
- **MEZCLA** no tiene composicion quimica explicita en el dataset.
- **Assessment** no usa umbrales completos **H, HH, V0** en el Excel → **fallback** de percentiles.
- **SHAP** / permutation importance **no** prueban causalidad.

## 12. Conclusiones y recomendaciones

- La **firma vibratoria** permite clasificar el **Batch** con buen desempeno en CV; **XGBoost** es el modelo seleccionado en la comparativa.
- Las clases son **estadisticamente diferenciables** (PERMANOVA) y hay senales de **dispersion interna distinta** (PERMDISP).
- **Recomendaciones:** lineas base **diferenciadas por batch**; completar **H, HH y V0** para un assessment mas normativo; validar con **mas dias/campanas** antes de uso operacional real.

## Checklist de reproducibilidad

Tabla generada en la siguiente celda: existencia de archivos clave y conteos esperados (sin ejecutar pipeline desde el notebook).

In [ ]:
def _exists(rel: Path) -> bool:
    return rel.is_file()

def _rows(rel: Path) -> int:
    return len(pd.read_csv(rel)) if rel.is_file() else -1

chk = [
    ("data/dashboard/dashboard_kpis.csv", _exists(DASHBOARD_DATA / "dashboard_kpis.csv"), ""),
    ("data/processed/bpc_windowed_features.csv", _exists(DATA_PROCESSED / "bpc_windowed_features.csv"), ""),
    ("outputs/models/model_final_xgboost.pkl", _exists(OUTPUTS_MODELS / "model_final_xgboost.pkl"), ""),
    ("dashboard/app.py", _exists(PROJECT_ROOT / "dashboard" / "app.py"), ""),
    ("data/dashboard/dashboard_predictions.csv (1451 filas)", _rows(DASHBOARD_DATA / "dashboard_predictions.csv") == 1451, f"n={_rows(DASHBOARD_DATA / 'dashboard_predictions.csv')}"),
    ("data/dashboard/dashboard_feature_importance.csv (120)", _rows(DASHBOARD_DATA / "dashboard_feature_importance.csv") == 120, f"n={_rows(DASHBOARD_DATA / 'dashboard_feature_importance.csv')}"),
    ("data/dashboard/dashboard_model_metrics.csv (5 modelos)", _rows(DASHBOARD_DATA / "dashboard_model_metrics.csv") == 5, f"n={_rows(DASHBOARD_DATA / 'dashboard_model_metrics.csv')}"),
    ("data/dashboard/dashboard_condition_index_by_batch.csv (3)", _rows(DASHBOARD_DATA / "dashboard_condition_index_by_batch.csv") == 3, f"n={_rows(DASHBOARD_DATA / 'dashboard_condition_index_by_batch.csv')}"),
]
df_chk = pd.DataFrame(chk, columns=["item", "ok", "nota"])
display(df_chk)
